# LEGO Collector Value Engine
## Finding best valued LEGO sets for their price
 
Every year, LEGO releases hundreds of sets with polished marketing and premium pricing.
But which sets actually deliver the best value for a collector's money?
 
This pipeline ingests raw LEGO product data and builds a mathematical scoring engine
that ranks every set on objective metrics - price, piece count, average rating, and amount of reviews - to find the sets worth buying and the ones worth skipping.
 
**Pipeline Architecture:** Bronze (raw) -> Silver (clean + scored) -> Gold (ranked analysis)
 
| Layer    | Purpose                                      |
|----------|----------------------------------------------|
| Bronze   | Raw ingestion - preserve data as received    |
| Silver   | Clean data + calculate collector value scores|
| Gold     | Ranked analytics tables ready for decisions  |
| ML       | KMeans segmentation of the collector market  |

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
 
print("Libraries loaded")

## Bronze Layer - Raw Ingestion

In [0]:
df_bronze = spark.table("workspace.default.lego_bronze_upload")
df_bronze = df_bronze.withColumn("ingested_at", F.current_timestamp())
df_bronze.write.format("delta").mode("overwrite").saveAsTable("lego_bronze")
 
print(f"Bronze layer complete. {df_bronze.count()} rows ingested.")
display(df_bronze.limit(5))

## Silver Layer - Data Cleansing and Score Calculation
 
Remove the following:
- Sets with missing prices
- Sets with missing ratings
- Sets with zero piece counts (prevent division by zero in our metrics)
  
Three collector metrics are then calculated from the clean data:
 
**Smart Value Index** = (rating x pieces) / price
The primary score. Combines a set's rating, piece count, and price into a single score that estimates how much value a collector receives for every dollar spent.
A high-piece, high-rated, low-price would score the highest.
 
**Collector Potential Score** = (rating x reviews) / price
Measures community-validated value. Sets with strong ratings AND high review
counts signal broad collector consensus, not just a few enthusiastic buyers.
 
**Price Efficiency** = pieces / price
Pure brick-per-dollar. Useful for builders who optimize for parts count.

In [0]:
# SILVER LAYER - cleanse + calculate value metrics
spark.sql("""
    CREATE OR REPLACE TABLE lego_silver AS
    SELECT
        set_id,
        set_name,
        theme,
        piece_count,
        ROUND(CAST(price_usd AS DOUBLE), 2) AS price_usd,
        ROUND(CAST(rating_value AS DOUBLE), 2) AS rating_value,
        review_count,
        release_year,
        ROUND((CAST(rating_value AS DOUBLE) * piece_count) / CAST(price_usd AS DOUBLE), 2) AS smart_value_index,
        ROUND((CAST(rating_value AS DOUBLE) * review_count) / CAST(price_usd AS DOUBLE), 2) AS collector_potential_score,
        ROUND(piece_count / CAST(price_usd AS DOUBLE), 2) AS price_efficiency,
        current_timestamp() AS processed_at
    FROM lego_bronze
    WHERE
        price_usd IS NOT NULL
        AND rating_value IS NOT NULL
        AND piece_count > 0
""")

silver_count = spark.table("lego_silver").count()
print(f"Silver layer complete. {silver_count} sets scored. {100 - silver_count} removed for missing data.")
display(spark.table("lego_silver").limit(5))

## Delta Time Travel - Auditing the Historical Record
 
Every change to the
dataset is versioned. If LEGO adjusts pricing (which they do), we can audit
what a set cost at any point in time and how that changed its value score.
 
Here we simulate a price correction on the Millennium Falcon and then query
the original Version 0 to compare.
![image_1780200109904.png](./image_1780200109904.png "image_1780200109904.png")

In [0]:
spark.sql("""
    UPDATE lego_silver
    SET price_usd = 799.99,
        smart_value_index = ROUND((rating_value * piece_count) / 799.99, 2)
    WHERE set_name = 'Millennium Falcon'
""")
 
print("Price correction applied to Millennium Falcon.")
 
print("\nOriginal record (Version 0):")
display(spark.sql("""
    SELECT set_name, price_usd, smart_value_index
    FROM lego_silver VERSION AS OF 0
    WHERE set_name = 'Millennium Falcon'
"""))
 
print("\nCurrent record (after correction):")
display(spark.sql("""
    SELECT set_name, price_usd, smart_value_index
    FROM lego_silver
    WHERE set_name = 'Millennium Falcon'
"""))
 
print("\nFull version history:")
display(spark.sql("DESCRIBE HISTORY lego_silver"))

## Gold Layer - Collector Intelligence
 
Three purpose-built tables that answer the questions every collector actually asks.
 
**Table 1 - Theme Insight:** Which LEGO themes consistently deliver value?
Are Star Wars sets overpriced relative to their rating? Does Technic punch above
its weight for builders? This table answers it by theme.
 
**Table 2 - The Undervalued 10:** The ten sets where the Smart Value Index is
highest - the mathematically best buys in the current catalog.
 
**Table 3 - Premium Collector Sets:** High-price sets that still earn it -
the sets worth the premium because the community has validated them.

In [0]:
# Themes
spark.sql("""
    CREATE OR REPLACE TABLE theme_analytics_gold AS
    SELECT
        theme,
        COUNT(*)                                  AS total_sets,
        ROUND(AVG(price_usd), 2)                  AS avg_price,
        ROUND(AVG(rating_value), 2)               AS avg_rating,
        ROUND(AVG(smart_value_index), 2)          AS avg_value_index,
        ROUND(AVG(collector_potential_score), 2)  AS avg_collector_score,
        ROUND(MIN(price_usd), 2)                  AS min_price,
        ROUND(MAX(price_usd), 2)                  AS max_price
    FROM lego_silver
    GROUP BY theme
    ORDER BY avg_value_index DESC
""")
print("theme_analytics_gold created.")
display(spark.table("theme_analytics_gold"))
 
# Top 10 undervalued sets
spark.sql("""
    CREATE OR REPLACE TABLE undervalued_sets_gold AS
    SELECT
        set_name,
        theme,
        price_usd,
        rating_value,
        piece_count,
        smart_value_index,
        collector_potential_score,
        price_efficiency,
        RANK() OVER (ORDER BY smart_value_index DESC) AS value_rank
    FROM lego_silver
    ORDER BY smart_value_index DESC
    LIMIT 10
""")
print("undervalued_sets_gold created.")
display(spark.table("undervalued_sets_gold"))
 
# Premium sets: expensive sets that earn their price
spark.sql("""
    CREATE OR REPLACE TABLE collector_segment_gold AS
    SELECT
        set_name,
        theme,
        price_usd,
        rating_value,
        review_count,
        collector_potential_score,
        smart_value_index
    FROM lego_silver
    WHERE price_usd >= 200
      AND rating_value >= 4.7
    ORDER BY collector_potential_score DESC
""")
print("collector_segment_gold created.")
display(spark.table("collector_segment_gold"))

## Machine Learning - Collector Market Segmentation

Not all LEGO buyers are the same. A KMeans clustering model groups the 91
scoreable sets into four distinct market segments based on price, rating,
piece count, and Smart Value Index.

By letting the AI analyze the mathematical relationships, we can map the 
catalog to four natural collector profiles:

1. **Premium Collector:** Flagship models with massive piece counts, near-perfect ratings, and premium price tags.
2. **High Value (The Hidden Gems):** Sets that heavily over-deliver on piece count and community rating relative to their cost.
3. **Casual Buyers:** Standard, mid-range retail sets that form the bulk of a typical consumer's collection.
4. **Budget-Friendly:** Highly accessible, low-cost sets that still maintain solid community approval.

This answers a question no dashboard alone can: are there natural clusters
in the market, and which segment does each set belong to? The result is a
data-driven segmentation of the entire LEGO catalog.

In [0]:
df_ml = spark.table("lego_silver").select(
    F.col("set_id"), 
    F.col("set_name"), 
    F.col("theme"),
    F.col("price_usd").cast("double"), 
    F.col("rating_value").cast("double"), 
    F.col("piece_count").cast("double"), 
    F.col("smart_value_index").cast("double")
).dropna()

assembler = VectorAssembler(
    inputCols=["price_usd", "rating_value", "piece_count", "smart_value_index"],
    outputCol="features_raw"
)
scaler = StandardScaler(
    inputCol="features_raw", outputCol="features",
    withStd=True, withMean=True
)
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=4, seed=42)

pipeline = Pipeline(stages=[assembler, scaler, kmeans])
model = pipeline.fit(df_ml)
df_clustered = model.transform(df_ml)

# Check cluster characteristics before labeling
display(df_clustered.groupBy("cluster").agg(
    F.count("*").alias("count"),
    F.round(F.avg("price_usd"), 2).alias("avg_price"),
    F.round(F.avg("rating_value"), 2).alias("avg_rating"),
    F.round(F.avg("smart_value_index"), 2).alias("avg_value_index")
).orderBy("avg_price"))

## Cluster Labeling
 
The result was four distinct groups of products that represent different types of buyers, including Budget-Friendly, Premium Collectors, High Value set shoppers, and Casual Buyers.

In [0]:
df_labeled = df_clustered.withColumn("segment", F.expr("""
    CASE cluster
        WHEN 0 THEN 'Budget-Friendly'
        WHEN 1 THEN 'Premium Collector'
        WHEN 2 THEN 'High Value'
        WHEN 3 THEN 'Casual Buyers'
        ELSE 'Unknown'
    END
"""))

df_labeled.select(
    "set_id", "set_name", "theme", "price_usd", "rating_value",
    "piece_count", "smart_value_index", "cluster", "segment"
).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ml_segments_gold")

print("Clustering complete. Segment summary:")
display(spark.sql("""
    SELECT segment, COUNT(*) AS set_count,
           ROUND(AVG(price_usd), 2) AS avg_price,
           ROUND(AVG(rating_value), 2) AS avg_rating,
           ROUND(AVG(smart_value_index), 2) AS avg_value_index
    FROM ml_segments_gold
    GROUP BY segment
    ORDER BY avg_price DESC
"""))

## Validation

In [0]:
tables = [
    "lego_bronze",
    "lego_silver",
    "theme_analytics_gold",
    "undervalued_sets_gold",
    "collector_segment_gold",
    "ml_segments_gold"
]
 
print("Pipeline Summary:\n")
for t in tables:
    try:
        count = spark.table(t).count()
        print(f"  {t:<35} {count} rows")
    except Exception as e:
        print(f"  {t:<35} ERROR: {e}")
 
print("\nPipeline complete.")